# 03 - Verify MPS (Apple Silicon GPU) Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import platform
import torch

from config import get_device, RANDOM_SEED

## Environment info

In [2]:
print(f"Python version:       {sys.version.split()[0]}")
print(f"Platform:             {platform.platform()}")
print(f"Machine:              {platform.machine()}  (arm64 = Apple Silicon)")
print(f"PyTorch version:      {torch.__version__}")
print()
print(f"MPS built into torch: {torch.backends.mps.is_built()}")
print(f"MPS available:        {torch.backends.mps.is_available()}")
print(f"CUDA available:       {torch.cuda.is_available()}")
print()

device = get_device()
print(f"Selected device:      {device}")

Python version:       3.11.14
Platform:             macOS-15.3-arm64-arm-64bit
Machine:              arm64  (arm64 = Apple Silicon)
PyTorch version:      2.11.0

MPS built into torch: True
MPS available:        True
CUDA available:       False

Selected device:      mps


## Run a small tensor operation on MPS

Matrix multiplication on GPU as a functional test.

In [3]:
import time

# Build two matrices on CPU
a_cpu = torch.randn(2048, 2048)
b_cpu = torch.randn(2048, 2048)

# --- CPU benchmark ---
t0 = time.perf_counter()
c_cpu = a_cpu @ b_cpu
cpu_time = time.perf_counter() - t0
print(f"CPU matmul (2048x2048): {cpu_time * 1000:.1f} ms")

# --- MPS benchmark (if available) ---
if device.type == "mps":
    a_mps = a_cpu.to(device)
    b_mps = b_cpu.to(device)

    # Warm-up: first MPS call compiles kernels
    _ = a_mps @ b_mps
    torch.mps.synchronize()

    t0 = time.perf_counter()
    c_mps = a_mps @ b_mps
    torch.mps.synchronize()  # wait for GPU to finish
    mps_time = time.perf_counter() - t0
    print(f"MPS matmul (2048x2048): {mps_time * 1000:.1f} ms")
    print(f"Speedup vs CPU: {cpu_time / mps_time:.1f}×")

    # Verify results match
    max_diff = (c_cpu - c_mps.cpu()).abs().max().item()
    print(f"Max abs diff CPU vs MPS: {max_diff:.2e}  (should be < 1e-2)")

CPU matmul (2048x2048): 30.1 ms
MPS matmul (2048x2048): 11.0 ms
Speedup vs CPU: 2.7×
Max abs diff CPU vs MPS: 0.00e+00  (should be < 1e-2)


## Tiny CNN sanity check

Dummy CNN forward + backward on MPS, using a batch that matches the eventual input shape `(batch, 1, 8, 50)`.

In [4]:
import torch.nn as nn

torch.manual_seed(RANDOM_SEED)

# Dummy model: 2 conv blocks, same order of magnitude as the real SCNN will be
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(32, 7),  # 7 gesture classes
).to(device)

# Dummy batch: 64 windows of shape (1, 8, 50) - matches real data layout
x = torch.randn(64, 1, 8, 50, device=device)
y = torch.randint(0, 7, (64,), device=device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for step in range(5):
    optimizer.zero_grad()
    logits = model(x)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()
    print(f"Step {step}: loss = {loss.item():.4f}")

print(f"\n✓ Forward + backward pass on {device} works.")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

Step 0: loss = 1.9551
Step 1: loss = 1.9453
Step 2: loss = 1.9364
Step 3: loss = 1.9283
Step 4: loss = 1.9209

✓ Forward + backward pass on mps works.
  Parameters: 5,031
